# 面试问题：Agent 怎样把计划表示为 DAG，并安全地并行调度？

可以直接复述的回答是：第一，把每个步骤建模为带依赖、资源、超时和幂等键的节点。第二，调度器只能释放依赖全部成功的节点。第三，互不依赖的步骤可以并行，但同一独占资源仍需串行。第四，计划执行前必须验证缺失依赖和环。第五，失败节点应阻断后继节点，而不是让整张图继续写入外部系统。第六，用 makespan、资源等待和关键路径比较 DAG 与顺序执行，而不是只说并行更快。下面用一次生产故障恢复计划演示。

## 真实案例：支付 API 故障恢复计划

输入模拟值班工程师批准后的六个操作：冻结流量、数据库快照、修复 API、冒烟验证、恢复流量和通知客户。字段与真实编排器一致，包括任务编号、依赖、预计分钟数和独占资源。数据来自脱敏教学事件，不连接真实基础设施；耗时是确定性估计，不能代表线上执行时延。

In [1]:
tasks = [  # 定义六个具有依赖和资源约束的故障恢复任务
    {"id": "freeze", "name": "冻结支付写流量", "deps": [], "minutes": 1, "resource": "gateway"},  # 首先阻止新的写请求进入
    {"id": "snapshot", "name": "创建数据库快照", "deps": ["freeze"], "minutes": 3, "resource": "database"},  # 在变更前保存可回滚状态
    {"id": "patch", "name": "部署 API 修复版本", "deps": ["freeze"], "minutes": 4, "resource": "application"},  # 快照与修复可使用不同资源并行
    {"id": "smoke", "name": "执行支付冒烟测试", "deps": ["snapshot", "patch"], "minutes": 2, "resource": "qa"},  # 只有两条前置分支都成功后才验证
    {"id": "restore", "name": "恢复支付写流量", "deps": ["smoke"], "minutes": 1, "resource": "gateway"},  # 验证成功后才能恢复入口
    {"id": "notify", "name": "发送客户恢复通知", "deps": ["restore"], "minutes": 1, "resource": "communications"},  # 最后发送带状态依据的通知
]  # 结束故障恢复计划定义
print("计划输入：id | 依赖 | 分钟 | 资源 | 操作")  # 展示调度器实际读取的结构化字段
for task in tasks:  # 逐项输出六个恢复任务
    print(f"{task['id']:8} | {','.join(task['deps']) or '-':15} | {task['minutes']} | {task['resource']:14} | {task['name']}")  # 用表格呈现依赖关系和业务语义


计划输入：id | 依赖 | 分钟 | 资源 | 操作
freeze   | -               | 1 | gateway        | 冻结支付写流量
snapshot | freeze          | 3 | database       | 创建数据库快照
patch    | freeze          | 4 | application    | 部署 API 修复版本
smoke    | snapshot,patch  | 2 | qa             | 执行支付冒烟测试
restore  | smoke           | 1 | gateway        | 恢复支付写流量
notify   | restore         | 1 | communications | 发送客户恢复通知


## Baseline / 基线：按列表顺序串行执行

顺序计划最容易实现，也不会违反当前列表中的依赖，但它让快照和 API 修复互相等待。我们先计算每个步骤的开始、结束时间和总耗时。

In [2]:
def sequential_schedule(items):  # 实现按输入顺序逐个执行的基线调度器
    clock = 0  # 从故障处置开始时刻零分钟计时
    rows = []  # 保存每个任务的开始与结束时间
    for item in items:  # 严格按照计划列表执行所有任务
        start = clock  # 当前任务只能等上一个任务结束
        end = start + item["minutes"]  # 根据预计耗时计算结束时刻
        rows.append({"id": item["id"], "start": start, "end": end, "resource": item["resource"]})  # 记录基线时间线
        clock = end  # 把串行时钟推进到当前任务末尾
    return rows  # 返回完整的顺序执行时间线
sequential_rows = sequential_schedule(tasks)  # 对六个故障恢复任务运行基线
print("顺序基线：id | start | end | resource")  # 输出每一步的基线时间窗口
for row in sequential_rows:  # 逐条展示顺序执行时间线
    print(f"{row['id']:8} | {row['start']:2} | {row['end']:2} | {row['resource']}")  # 让不必要的等待可被直接观察
print(f"顺序总耗时：{sequential_rows[-1]['end']} 分钟")  # 展示基线 makespan


顺序基线：id | start | end | resource
freeze   |  0 |  1 | gateway
snapshot |  1 |  4 | database
patch    |  4 |  8 | application
smoke    |  8 | 10 | qa
restore  | 10 | 11 | gateway
notify   | 11 | 12 | communications
顺序总耗时：12 分钟


## 核心实现：依赖就绪与资源占用共同决定开始时间

先做拓扑验证，再按依赖最晚结束时刻和资源可用时刻计算节点开始时间。这个离线调度器用于解释机制，不模拟任务时长抖动。

In [3]:
def dag_schedule(items):  # 从零实现依赖与独占资源感知的 DAG 调度
    by_id = {item["id"]: item for item in items}  # 建立任务编号到定义的索引
    missing = [(item["id"], dep) for item in items for dep in item["deps"] if dep not in by_id]  # 收集所有不存在的前置依赖
    if missing:  # 缺失依赖时计划不能开始执行
        raise ValueError(f"missing_dependency:{missing}")  # 返回可定位的计划合同错误
    pending = set(by_id)  # 初始化尚未排程的任务集合
    finished = {}  # 保存已排程任务的结束时刻
    resource_free = {}  # 保存每个独占资源的下一可用时刻
    rows = []  # 收集最终的 DAG 时间线
    while pending:  # 持续释放依赖已完成的节点
        ready = sorted(task_id for task_id in pending if all(dep in finished for dep in by_id[task_id]["deps"]))  # 找出当前可调度节点
        if not ready:  # 仍有任务但没有就绪节点说明存在依赖环
            raise ValueError(f"dependency_cycle:{sorted(pending)}")  # 阻止循环计划进入执行阶段
        for task_id in ready:  # 对同一拓扑层中的任务分别计算时间窗口
            item = by_id[task_id]  # 读取当前任务的完整定义
            dependency_end = max([finished[dep] for dep in item["deps"]], default=0)  # 获取所有前置节点的最晚结束时刻
            start = max(dependency_end, resource_free.get(item["resource"], 0))  # 同时满足依赖与独占资源约束
            end = start + item["minutes"]  # 计算当前任务的预计结束时刻
            rows.append({"id": task_id, "start": start, "end": end, "resource": item["resource"], "deps": item["deps"]})  # 写入可审计时间线
            finished[task_id] = end  # 记录结束时刻供后继任务使用
            resource_free[item["resource"]] = end  # 更新独占资源的可用时刻
            pending.remove(task_id)  # 从待排程集合移除当前任务
    return sorted(rows, key=lambda row: (row["start"], row["id"]))  # 按真实开始时间返回计划
dag_rows = dag_schedule(tasks)  # 对故障恢复计划执行 DAG 调度
print("DAG 时间线：id | start | end | deps")  # 输出核心算法生成的并行计划
for row in dag_rows:  # 逐条展示每个节点的时间窗口
    print(f"{row['id']:8} | {row['start']:2} | {row['end']:2} | {','.join(row['deps']) or '-'}")  # 显示快照与修复的并行关系


DAG 时间线：id | start | end | deps
freeze   |  0 |  1 | -
patch    |  1 |  5 | freeze
snapshot |  1 |  4 | freeze
smoke    |  5 |  7 | snapshot,patch
restore  |  7 |  8 | smoke
notify   |  8 |  9 | restore


## 失败案例与修正：有环计划会永久等待

错误计划让 smoke 依赖 restore，同时 restore 又依赖 smoke。天真的调度循环会一直等待；修正后的实现检测“pending 非空且 ready 为空”，在调用任何外部工具前拒绝计划。

In [4]:
cyclic_tasks = [dict(task) for task in tasks]  # 复制正常计划以构造不会污染原数据的反例
for task in cyclic_tasks:  # 查找需要注入错误依赖的冒烟任务
    if task["id"] == "smoke":  # 只修改 smoke 节点的依赖
        task["deps"] = ["snapshot", "patch", "restore"]  # 人为形成 smoke 与 restore 的依赖环
failure_message = ""  # 保存计划验证器返回的可读错误
try:  # 尝试调度带环的错误计划
    dag_schedule(cyclic_tasks)  # 在任何真实动作前执行计划验证
except ValueError as error:  # 捕获预期的依赖环合同错误
    failure_message = str(error)  # 保存错误内容用于审计和教学展示
repaired_rows = dag_schedule(tasks)  # 删除错误依赖后重新生成安全计划
print("失败计划检测：", failure_message)  # 展示原方案为何无法继续
print(f"修正后成功排程 {len(repaired_rows)} 个节点，未调用任何真实生产工具")  # 展示修正后的可执行状态


失败计划检测： dependency_cycle:['notify', 'restore', 'smoke']
修正后成功排程 6 个节点，未调用任何真实生产工具


## 结果表：顺序执行与 DAG 调度对照

In [5]:
sequential_makespan = max(row["end"] for row in sequential_rows)  # 计算顺序方案的最终结束时刻
dag_makespan = max(row["end"] for row in dag_rows)  # 计算 DAG 方案的最终结束时刻
saved_minutes = sequential_makespan - dag_makespan  # 量化并行调度节省的教学时间
parallel_pairs = [(left["id"], right["id"]) for left in dag_rows for right in dag_rows if left["id"] < right["id"] and left["start"] < right["end"] and right["start"] < left["end"]]  # 找出时间窗口重叠的任务对
print("方案 | makespan(分钟) | 节省(分钟)")  # 输出同一指标下的方案对照
print(f"sequential | {sequential_makespan} | 0")  # 展示串行基线总耗时
print(f"dag        | {dag_makespan} | {saved_minutes}")  # 展示 DAG 调度总耗时和节省量
print("并行窗口：", parallel_pairs)  # 明确指出哪些任务实际发生了重叠


方案 | makespan(分钟) | 节省(分钟)
sequential | 12 | 0
dag        | 9 | 3
并行窗口： [('patch', 'snapshot')]


## 结果解读

快照和 API 修复都依赖 freeze，却使用不同资源，因此在第 1 分钟同时开始；smoke 等较慢的 patch 完成后才释放。DAG 将教学 makespan 从 12 分钟降到 9 分钟，收益来自真实依赖结构而非跳过步骤。若耗时预测失准或资源容量大于一，当前离线模型需要升级为事件驱动调度器。

## 生产边界

生产编排必须增加任务超时、重试预算、幂等键、补偿动作、人工审批和动态资源容量；节点成功还要依据权威回读，而不是进程返回码。调度状态应持久化到事件存储，主备调度器通过租约避免重复释放节点。本例不执行 Kubernetes、数据库或支付操作，也未模拟随机时延。

## 最小回归测试

In [6]:
assert len(tasks) >= 5  # 保证案例包含足够多的真实计划节点
assert dag_makespan < sequential_makespan  # 保证 DAG 在本案例中确实减少总耗时
assert next(row for row in dag_rows if row["id"] == "snapshot")["start"] == 1  # 保证快照在冻结流量后立即开始
assert next(row for row in dag_rows if row["id"] == "patch")["start"] == 1  # 保证修复与快照处于同一并行层
assert failure_message.startswith("dependency_cycle")  # 保证依赖环在执行前被明确拒绝
